# 02 Data Cleaning & Standardisation
## BhavIndex - Mumbai & Pune Real Estate Price Analytics

### Objective
Execute reproducible data filtering, standardisation, text parsing, and outlier handling according to the project specifications.


In [1]:
import pandas as pd
import re

raw_df = pd.read_csv('../Data/Raw/mumbai_pune_properties.csv')

# Exclusion rule set
EXCLUDE_TERMS = ['office space', 'commercial', 'shop', 'showroom', 'industrial land', 'plot/land', 'godown', 'warehouse']

def get_exclusion_reason(row):
    title = str(row['Title']).lower()
    for term in EXCLUDE_TERMS:
        if term in title:
            return f"Non-residential property type: '{term}'"
    if pd.isna(row['BHK']) and not any(k in title for k in ['flat', 'apartment', 'villa', 'penthouse', 'studio']):
        return "Missing BHK and unclassified residential title"
    if row['Price_INR'] > 1e9:
        return f"Scraper corruption price anomaly (> 100 Crore / {row['Price_INR']})"
    if row['Price_INR'] < 1e5:
        return f"Implausibly low price (< 1 Lakh / {row['Price_INR']})"
    return None

raw_df['exclusion_reason'] = raw_df.apply(get_exclusion_reason, axis=1)

excluded_df = raw_df[raw_df['exclusion_reason'].notnull()].copy()
residential_df = raw_df[raw_df['exclusion_reason'].isnull()].copy().drop(columns=['exclusion_reason'])

print(f"Retained Residential Records: {len(residential_df)}")
print(f"Excluded Non-Residential / Corrupted: {len(excluded_df)}")


Retained Residential Records: 1799
Excluded Non-Residential / Corrupted: 244


### Save Interim Datasets


In [1]:
residential_df.to_csv('../Data/Interim/residential_properties.csv', index=False)
excluded_df.to_csv('../Data/Interim/excluded_properties.csv', index=False)
print("Saved interim datasets successfully.")


Saved interim datasets successfully.


### Standardisation & Cleaning
Standardize column names, trim whitespace, extract localities, and impute logical defaults.


In [1]:
# Standardize categorical values
residential_df['City'] = residential_df['City'].str.strip().str.title()
residential_df['Furnishing'] = residential_df['Furnishing'].fillna('Unfurnished')
residential_df['Status'] = residential_df['Status'].fillna('Ready to Move')

# Standardize bathrooms bounded logically by BHK
residential_df['Bathrooms'] = residential_df['Bathrooms'].fillna(residential_df['BHK'].clip(lower=1, upper=5))

# Calculate diagnostic price per sqft (NOT a feature for ML)
residential_df['price_per_sqft'] = (residential_df['Price_INR'] / residential_df['Area_sqft']).round(2)

residential_df.to_csv('../Data/Processed/properties_cleaned.csv', index=False)
print(f"Final Cleaned Dataset: {len(residential_df)} rows")


Final Cleaned Dataset: 1799 rows
